# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` fields.

### Dataset Source
The dataset source is defined via a Croissant schema URL:
<br>
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print("\nDataset Name:")
print(metadata.name)
print("\nDescription:")
print(metadata.description)
print("\nLicense:")
print(metadata.license)
print("\nCite as:")
print(metadata.citeAs)
print("\nKeywords:")
print(metadata.keywords)


## 2. Data Overview
Review available record sets, fields, columns, and their IDs. All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets by @id
record_sets = dataset.record_sets()

print("Available record sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '-')})")

# Optionally, print fields for each record set
for rs in record_sets:
    print(f"\nFields in record set {rs['@id']}:")
    fields = dataset.fields(record_set=rs['@id'])
    for fld in fields:
        print(f"    - {fld['@id']} (name: {fld.get('name', '-')}, dataType: {fld.get('dataType', '-')})")


## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. All references use record set and field `@id`s from the overview.

In [ ]:
# Gather all record set @id's in a list
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records from each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nFirst 5 rows of DataFrame for record set @id: {record_set_id}")
        print(df.head())
        print(f"Available columns (@id): {df.columns.tolist()}\n")

# For demonstration, select the first record set with data
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    example_df = dataframes[example_record_set_id]
    print(f"Columns for example record set @id: {example_record_set_id}:")
    print(example_df.columns.tolist())
else:
    example_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
Explore and process the loaded data by selecting numeric and grouping fields via their `@id`.

* Filtering records based on specific criteria
* Normalizing numeric fields
* Grouping by categorical fields (by `@id`)

Example operations:

In [ ]:
# EDA on the example DataFrame (record set)
if example_record_set_id is not None:
    df = dataframes[example_record_set_id]
    # List numeric fields (by @id)
    numeric_fields = []
    string_fields = []

    # Fetch field metadata to determine types
    fields_meta = dataset.fields(record_set=example_record_set_id)
    for f in fields_meta:
        # Prefer 'Integer' or 'Float' datatype
        dtype = f.get('dataType', '').lower()
        if dtype in ['integer', 'float', 'number']:
            # Column name is usually the @id
            if f['@id'] in df.columns:
                numeric_fields.append(f['@id'])
        else:
            if f['@id'] in df.columns:
                string_fields.append(f['@id'])

    # Select a numeric field for demonstration
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        # Set a threshold for filtering
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Group by a string/categorical field (@id)
        if string_fields:
            group_field_id = string_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (mean of numeric fields):")
            print(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")
else:
    print("No record sets with data available for EDA.")

## 5. Visualization
Visualize distributions or relationships between dataset fields (referenced by `@id`). Example visualization below.

In [ ]:
# Example visualization: Histogram and box plot of numeric field
if example_record_set_id is not None and numeric_fields:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1,2,2)
    df.boxplot(column=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id}")

    plt.tight_layout()
    plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The FAIR^2 dataset provides clinicopathological and molecular characteristics for second primary colorectal cancer survivors, enabling detailed biomarker and predictor studies.
* Data exploration via `mlcroissant` allows entity referencing by `@id`, ensuring reproducible and transparent workflows.
* The dataset contains well-defined record sets and fields (with standardized types), supporting clinical analyses such as stratification of MSI-H status or anatomical distributions.
* Visualization and EDA steps demonstrate core approaches for future modeling or more advanced analyses.

> **For further use:** Refer to additional field and record set `@id`s for custom modeling, and apply domain-specific transformations or filtering following clinical requirements.
